# BPE-tokenizer op TweetEval stance (Jeff)

Project 1, stap 2: hoe segmenteren we de tweets? Hier lopen we de BPE-stappen uit de tutorial door op onze eigen data.

## Setup

In [ ]:
import sys
import warnings
from collections import Counter, OrderedDict
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer

warnings.simplefilter(action="ignore", category=FutureWarning)

# notebook staat in notebooks/, de repo-root is één map hoger
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1
from part1.clslib import Encoder
from src.data import load_stance

results_folder = root / "results"

## Data

Alleen de **train**-split, zodat de tokenizer niets van de test-data ziet.

TODO: `#SemST` staat in elke tweet (artefact van de dataset), overwegen om dat te verwijderen.

In [ ]:
stance_dataset = load_stance("feminist")
train_texts = [t.replace("#SemST", "").strip() for t in stance_dataset["train"]["text"]]
print(f"{len(train_texts)} train-tweets; de eerste:\n{train_texts[0]}")

## BPE stap voor stap

### Pretokenisatie

In [ ]:
prepared_corpus = part1.bpe.prepare_corpus(train_texts)
prepared_corpus[:5]

### Initiële vocabulaire: alle karakters + UNK-teken

In [ ]:
char_vocab = {ch for token in prepared_corpus for ch in token}
vocab = char_vocab | {"\u2047"}  # '⁇' voor karakters die we later nog nooit gezien hebben
sorted(vocab)

### Pretokens tellen

In [ ]:
word_counts = Counter(tuple(word) for word in prepared_corpus)
word_counts.most_common(10)

### Paren tellen en het meest frequente paar kiezen

In [ ]:
pair_counts = Counter()
for symbols, frequency in word_counts.items():
    for left, right in zip(symbols, symbols[1:]):
        pair_counts[(left, right)] += frequency

# bij gelijke telling: lexicografische volgorde als tie-breaker
best_pair = min(pair_counts, key=lambda pair: (-pair_counts[pair], pair))
best_pair, pair_counts[best_pair]

### Eerste merge vastleggen

De volgorde van de merges is wat telt bij het segmenteren van nieuwe tekst.

In [ ]:
merges = OrderedDict()
merges[best_pair] = pair_counts[best_pair]
merges

## Tokenizer trainen

Hugging Face BPE (zoals in de tutorial), getraind op de train-split.

In [ ]:
vocab_size = 1000
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)

## Representaties

### Token indicators

1 als een token in de tweet voorkomt, anders 0.

In [ ]:
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector


token_indicator_encoder = TokenIndicator(tokenizer)
doc_repr = token_indicator_encoder(train_texts[0])
print(len(doc_repr))
print(doc_repr.sum())

### Bag-of-words

Zelfde als token indicators, maar met het aantal keer dat een token voorkomt.

In [ ]:
class BagOfWords(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of token counts"""
        token_ids = self.tokenizer.encode(x).ids

        # Same as TokenIndicator, but we keep duplicate ids and count them,
        vector = np.bincount(token_ids, minlength=len(self)).astype(float)
        return vector


bow_encoder = BagOfWords(tokenizer)
doc_repr = bow_encoder(train_texts[0])
print(len(doc_repr))
print(doc_repr.sum())
print(len(tokenizer.encode(train_texts[0]).ids))  # should equal the sum above

## Labelverdeling

Meerderheidsklasse "against" is ~49%: een model dat altijd "against" zegt haalt al 49% accuracy. Daarom gebruiken we **macro-F1**.

In [ ]:
Counter(stance_dataset["train"]["label"])  # 0=none, 1=against, 2=favor

## Experimenten: 2 representaties × 2 modellen

Cross-validation op de train-split (`make_cv_splits`), zelfde `max_iter` voor alle logistic-runs.

In [ ]:
from sklearn.metrics import f1_score
from src.data import LABEL_NAMES, make_cv_splits

outcome_space = part1.clslib.NamedClasses(LABEL_NAMES)
splits = make_cv_splits(stance_dataset)

encoders = {"indicator": token_indicator_encoder, "bow": bow_encoder}
models = ["logistic", "mlp"]

results = {}
for model in models:
    for enc_name, encoder in encoders.items():
        name = f"{model}_{enc_name}"
        results[name] = part1.experiment.cross_validation_experiment(
            data_splits=splits[0],
            outcome_space=outcome_space,
            encoder=encoder,
            model_option=model,
            loglin_args={"max_iter": 1000},
            save_to=str(results_folder / name),
        )

### Macro-F1 per systeem

In [ ]:
def macro_f1_per_fold(result, cv_splits):
    return [
        f1_score([label for _, label in cv_splits[fold]["validation"]], result["predictions"][fold], average="macro")
        for fold in cv_splits
    ]

f1_scores = {name: macro_f1_per_fold(result, splits[0]) for name, result in results.items()}

for name, scores in f1_scores.items():
    print(f"{name:20s} macro-F1 = {np.mean(scores):.3f}  (per fold: {np.round(scores, 3)})")